In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#df = pd.read_csv()

In [0]:
data_path = "dbfs:/Volumes/workspace/default/capstoneproject/S&P500_Data_Clean/SP500_with_indicators/"

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    DateType,
    DoubleType,
    StringType
)

In [0]:
df = spark.read.format("delta").load(data_path)

df = df.withColumn("Date", F.to_date("Date"))

In [0]:
#Trend filter: Close > SMA_200

#Entry: RSI < 40

#Exit: +2%, -2%, or 7 days

df_signal_close_SMA200 = (
    df.withColumn(
        "entry_signal",
        F.when(
            (F.col("Close") > F.col("SMA_200")) & 
            (F.col("RSI_14") < F.lit(40)),
            F.lit(1)
        ).otherwise(F.lit(0))   
    )
    #.withColumn(
        #"exit_signal",
        #F.when(F.col("SMA_20") < F.col("SMA_50"), F.lit(1)).otherwise(F.lit(0))
    #)
)

In [0]:
#Entry: RSI < 40

#Exit: +2%, -2%, or 7 days

df_signal_RSI40 = (
    df.withColumn(
        "entry_signal",
        F.when(
            (F.col("RSI_14") < F.lit(40)),
            F.lit(1)
        ).otherwise(F.lit(0))   
    )
    #.withColumn(
        #"exit_signal",
        #F.when(F.col("SMA_20") < F.col("SMA_50"), F.lit(1)).otherwise(F.lit(0))
    #)
)

In [0]:
#Here we are creating our entry signal to enter a long position based on the following conditions:
#- SMA_20 > SMA_50 
#- RSI_14 > 50

#Our exit signal is when the SMA_20 crosses below the SMA_50 
#SMA_20 <SMA_50

df_signal_SMA20_50 = (
    df.withColumn(
        "entry_signal",
        F.when(
            (F.col("SMA_20") > F.col("SMA_50")),
            F.lit(1)
        ).otherwise(F.lit(0))   
    )
    .withColumn(
        "exit_signal",
        F.when(F.col("SMA_20") < F.col("SMA_50"), F.lit(1)).otherwise(F.lit(0))
    )
)

In [0]:
#Here we are creating our entry signal to enter a long position based on the following conditions:
#- SMA_20 > SMA_50 > SMA_200
#- RSI_14 > 50

#Our exit signal is when the SMA_20 crosses below the SMA_50 
#SMA_20 <SMA_50

df_signal_SMA20_50_200 = (
    df.withColumn(
        "entry_signal",
        F.when(
            (F.col("SMA_20") > F.col("SMA_50")) & 
            (F.col("SMA_50") > F.col("SMA_200")) &
            (F.col("RSI_14") > F.lit(50)),
            F.lit(1)
        ).otherwise(F.lit(0))   
    )
    .withColumn(
        "exit_signal",
        F.when(F.col("SMA_20") < F.col("SMA_50"), F.lit(1)).otherwise(F.lit(0))
    )
)

In [0]:
trades_path = "dbfs:/Volumes/workspace/default/capstoneproject/S&P500_Data_Clean/Trades"

In [0]:
trades_schema = StructType([
    StructField("series_id", StringType(), True),
    StructField("strategy_id", StringType(), True),
    StructField("entry_date", DateType(), True),
    StructField("entry_price", DoubleType(), True),
    StructField("exit_date", DateType(), True),
    StructField("exit_price", DoubleType(), True),
    StructField("return", DoubleType(), True),
    StructField("mae", DoubleType(), True),
    StructField("reason", StringType(), True),
    StructField("pnl_price", DoubleType(), True),
    StructField("cumulative_profit", DoubleType(), True)
])

In [0]:
STOP_LOSS_PCT = 0.02
TAKE_PROFIT_PCT = 0.02

In [0]:
def backtest_one_series(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values("Date").reset_index(drop=True)

    # Pull group identifiers from the incoming grouped pandas frame
    series_id = str(pdf["series_id"].iloc[0])
    strategy_id = str(pdf["strategy_id"].iloc[0])

    shares = 1
    trades = []

    in_pos = False
    entry_price = None
    entry_date = None
    mae = 0.0

    i = 0
    n = len(pdf)

    while i < n - 1:
        row = pdf.iloc[i]

        # Enter on next day's open after a signal
        if not in_pos:
            if int(row["entry_signal"]) == 1:
                entry_row = pdf.iloc[i + 1]
                in_pos = True
                entry_price = float(entry_row["Open"])
                entry_date = pd.to_datetime(entry_row["Date"]).date()
                mae = 0.0
                i += 1
                continue

        if in_pos:
            row = pdf.iloc[i]

            day_high = float(row["High"])
            day_low = float(row["Low"])
            exit_date = pd.to_datetime(row["Date"]).date()

            # MAE = worst intratrade drawdown from entry
            mae = min(mae, (day_low - entry_price) / entry_price)

            stop_price = entry_price * (1 - STOP_LOSS_PCT)
            take_price = entry_price * (1 + TAKE_PROFIT_PCT)

            exit_reason = None
            exit_price = None

            # If both happen same day, this code gives priority to stop loss
            if day_low <= stop_price:
                exit_reason = "stop_loss"
                exit_price = stop_price
            elif day_high >= take_price:
                exit_reason = "take_profit"
                exit_price = take_price

            if exit_reason is not None:
                ret = (exit_price - entry_price) / entry_price
                pnl = (exit_price - entry_price) * shares

                trades.append({
                    "series_id": series_id,
                    "strategy_id": strategy_id,
                    "entry_date": entry_date,
                    "entry_price": float(entry_price),
                    "exit_date": exit_date,
                    "exit_price": float(exit_price),
                    "return": float(ret),
                    "mae": float(mae),
                    "reason": str(exit_reason),
                    "pnl_price": float(pnl),
                    "cumulative_profit": None
                })

                in_pos = False
                entry_price = None
                entry_date = None
                mae = 0.0

        i += 1

    # Return empty dataframe with correct columns if no trades
    if not trades:
        return pd.DataFrame(columns=[
            "series_id",
            "strategy_id",
            "entry_date",
            "entry_price",
            "exit_date",
            "exit_price",
            "return",
            "mae",
            "reason",
            "pnl_price",
            "cumulative_profit"
        ])

    return pd.DataFrame(trades)

In [0]:
%sql
select * from sp500_with_indicators

In [0]:
def createTable(df_signal, strat_name):
    series_id = "S&P500"

    # Keep only needed columns and add identifiers
    df_two = (
        df_signal
        .withColumn("series_id", F.lit(series_id))
        .withColumn("strategy_id", F.lit(strat_name))
        .select(
            "series_id",
            "strategy_id",
            "Date",
            "Close",
            "High",
            "Low",
            "Open",
            "Volume",
            "prev_close",
            "prev_volume",
            "ret_1d",
            "log_ret_1d",
            "gap_ret",
            "mom_5d",
            "mom_10d",
            "mom_20d",
            "SMA_20",
            "SMA_50",
            "SMA_200",
            "vol_20d",
            "vol_20d_ann",
            "Vol_SMA_20",
            "rel_volume_20",
            "OBV",
            "VWAP",
            "CMF_20",
            "close_vs_sma20",
            "close_vs_sma50",
            "close_vs_sma200",
            "sma20_vs_sma50",
            "sma50_vs_sma200",
            "sma20_vs_sma200",
            "zclose_20",
            "EMA_12",
            "EMA_26",
            "MACD",
            "MACD_signal",
            "MACD_hist",
            "RSI_14",
            "ATR_14",
            "BB_mid_20",
            "BB_std_20",
            "BB_upper_20",
            "BB_lower_20",
            "BB_width_20",
            "BB_pctB_20",
            "stoch_k_14",
            "stoch_d_3",
            "entry_signal"
        )
    )

    # Run grouped backtest
    trades_spark = (
        df_two
        .groupBy("series_id", "strategy_id")
        .applyInPandas(backtest_one_series, schema=trades_schema)
    )

    # Running cumulative profit by exit date
    window_spec = (
        Window
        .partitionBy("series_id", "strategy_id")
        .orderBy("exit_date")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )

    trades_spark = trades_spark.withColumn(
        "cumulative_profit",
        F.sum("pnl_price").over(window_spec)
    )

    # Write Delta files
    trades_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(trades_path)

    trades_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"default.capstoneTrades_{strat_name}")

    return trades_spark

In [0]:
createTable(df_signal_close_SMA200, "CloseSMA200_V2")

In [0]:
createTable(df_signal_RSI40, "rsi40_V2")

In [0]:
createTable(df_signal_SMA20_50, "sma20_50_V2")

In [0]:
createTable(df_signal_SMA20_50_200, "sma20_50_200_V2")